In [0]:
data = [(1, "Tim", 24, "Kerala", "India"),
        (2, "Asman", 26, "Kerala", "India")]
df1 = spark.createDataFrame(data, ["emp_id", "name", "age", "state", "country"])
df1.show()

data2 = [(1, "Tim", 24, "Comcity"),
         (2, "Asman", 26, "bimcity")]
df2 = spark.createDataFrame(data2, ["emp_id", "name", "age", "address"])
df2.show()

In [0]:
df1.join(df2,'emp_id',"inner").select(df1["*"],df2["address"]).show()

In [0]:
data = [
    ("SEA", "SF", 300),
    ("CHI", "SEA", 2000),
    ("SF", "SEA", 300),
    ("SEA", "CHI", 2000),
    ("SEA", "LND", 500),
    ("LND", "SEA", 500),
    ("LND", "CHI", 1000),
    ("CHI", "LND", 180)]
df = spark.createDataFrame(data, ["from", "to", "dist"])
df.show()


In [0]:
df.createOrReplaceTempView("df")

In [0]:
%sql
select ff.from,ff.to,(f.dist+ff.dist) as sum from df f
join df ff on(f.from=ff.to and f.to=ff.from and f.from>ff.from)

In [0]:
result_df = spark.sql("""
    select ff.from, ff.to, (f.dist + ff.dist) as sum
    from df f
    join df ff on (f.from = ff.to and f.to = ff.from and f.from > ff.from)
""")
result_df.write.saveAsTable("bank.dst.route")

In [0]:
%sql
select * from bank.dst.route

In [0]:
%sql
describe formatted bank.dst.route

In [0]:
%sql
describe history bank.dst.route

In [0]:
%sql
describe extended bank.dst.route

In [0]:
data = [("A", "AA"), ("B", "BB"), ("C", "CC"), ("AA", "AAA"), ("BB", "BBB"), ("CC", "CCC")]

df = spark.createDataFrame(data, ["child", "parent"])
df.show()


In [0]:
df1 = df.alias("df1")
df = df.alias("df")


In [0]:
final=df.join(df1, df.parent == df1.child).select(df.child, df.parent, df1.parent.alias("grandparent"))
final.show()


In [0]:
from datetime import datetime, timedelta
from pyspark.sql.functions import col, lit, current_timestamp

# Initial orders data
orders_data = [
    (1, "CUST001", "Product A", 100.0, "Pending"),
    (2, "CUST002", "Product B", 200.0, "Shipped"),
    (3, "CUST003", "Product C", 150.0, "Pending")
]

orders_df = spark.createDataFrame(orders_data, ["order_id", "customer_id", "product", "amount", "status"])
orders_df.show()

In [0]:
# Create SCD2 target table with additional columns:
# - start_date: when this version became effective
# - end_date: when this version expired (null for current records)
# - is_current: flag indicating if this is the current version

# Drop table if exists to recreate with correct schema
spark.sql("DROP TABLE IF EXISTS bank.dst.orders_scd2")

spark.sql("""CREATE TABLE bank.dst.orders_scd2 (
    order_id BIGINT,
    customer_id STRING,
    product STRING,
    amount DOUBLE,
    status STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
) USING DELTA""")

print("SCD2 target table created successfully")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Initial load: add start_date, set end_date to null, and is_current to true
# Explicitly cast order_id to bigint to match table schema
initial_load = orders_df \
    .withColumn("order_id", col("order_id").cast("bigint")) \
    .withColumn("start_date", current_timestamp()) \
    .withColumn("end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

initial_load.write.format("delta").mode("overwrite").saveAsTable("bank.dst.orders_scd2")

print("Initial load completed")
spark.sql("SELECT * FROM bank.dst.orders_scd2").show(truncate=False)

In [0]:
from pyspark.sql.functions import col

# Updated orders data - order 1 status changed, order 2 amount changed, order 4 is new
updated_orders_data = [
    (1, "CUST001", "Product A", 100.0, "Shipped"),     # Status changed from Pending to Shipped
    (2, "CUST002", "Product B", 250.0, "Delivered"),  # Amount and status changed
    (3, "CUST003", "Product C", 150.0, "Pending"),    # No change
    (4, "CUST004", "Product D", 300.0, "Pending")     # New order
]

updated_orders_df = spark.createDataFrame(updated_orders_data, ["order_id", "customer_id", "product", "amount", "status"]) \
    .withColumn("order_id", col("order_id").cast("bigint"))
print("Updated orders:")
updated_orders_df.show()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit, col

# Read the target SCD2 table
target_table = DeltaTable.forName(spark, "bank.dst.orders_scd2")

# Prepare source with SCD2 columns
source_df = updated_orders_df \
    .withColumn("start_date", current_timestamp()) \
    .withColumn("end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

# Create a temp view for the source
source_df.createOrReplaceTempView("orders_source")

# Step 1: Expire old records that have changes
target_table.alias("target").merge(
    source_df.alias("source"),
    "target.order_id = source.order_id AND target.is_current = true"
).whenMatchedUpdate(
    condition="""
        target.customer_id <> source.customer_id OR
        target.product <> source.product OR
        target.amount <> source.amount OR
        target.status <> source.status
    """,
    set={
        "end_date": current_timestamp(),
        "is_current": lit(False)
    }
).execute()

# Step 2: Insert new versions for changed records and completely new records
target_table.alias("target").merge(
    source_df.alias("source"),
    "target.order_id = source.order_id AND target.is_current = true"
).whenNotMatchedInsert(
    values={
        "order_id": "source.order_id",
        "customer_id": "source.customer_id",
        "product": "source.product",
        "amount": "source.amount",
        "status": "source.status",
        "start_date": "source.start_date",
        "end_date": "source.end_date",
        "is_current": "source.is_current"
    }
).execute()

print("SCD2 merge completed successfully")

In [0]:
%sql
SELECT 
    order_id,
    customer_id,
    product,
    amount,
    status,
    start_date,
    end_date,
    is_current
FROM bank.dst.orders_scd2
ORDER BY order_id, start_date

In [0]:
%sql
-- Query to get only current/active records
SELECT 
    order_id,
    customer_id,
    product,
    amount,
    status,
    start_date
FROM bank.dst.orders_scd2
WHERE is_current = true
ORDER BY order_id